# Base: snippets + fixations + survey

In [1]:
from pathlib import Path

import numpy as np 
import pandas as pd 

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = next(i for i in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if i.name == "EoS-tosem2026")
METRICS_UPDATE_DIR = NOTEBOOK_DIR.parent
PROJECT_DIR = METRICS_UPDATE_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
SUPPLEMENTARY_DIR = PROJECT_DIR / "supplementary"
OUTPUT_DIR = METRICS_UPDATE_DIR / "data"

SNIPPETS_PATH = OUTPUT_DIR / "code_snippets.csv"
# AOI labels (chunk, inside/outside, bottom-up/top-down) re-derived by
# metrics_update/analysis/relabel_aoi.py from data/all_exp_fixations_original2_cleaned.csv
# (fixes participants 10 and 27 and snippet 13).
FIXATIONS_PATH = OUTPUT_DIR / "fixations_aoi_relabeled.csv"
ANSWERS_PATH = SUPPLEMENTARY_DIR / "answers_experiments.tsv"
OUTPUT_PATH = OUTPUT_DIR / "base_fixations_design_survey.csv"

def draw_line(sep="=", n=90):
    print(sep * n)

def print_head(description, sep_1="=", sep_2="-", n=90):
    draw_line(sep_1, n)
    print(description)
    draw_line(sep_2, n)

def show_relative_path(label: str, path: Path, root_dir: Path=ROOT_DIR):
    print(f"{label}: {path.relative_to(root_dir)}")

print_head("Paths")
show_relative_path("SNIPPETS_PATH", SNIPPETS_PATH)
show_relative_path("FIXATIONS_PATH", FIXATIONS_PATH)
show_relative_path("ANSWERS_PATH", ANSWERS_PATH)
show_relative_path("OUTPUT_PATH", OUTPUT_PATH)

Paths
------------------------------------------------------------------------------------------
SNIPPETS_PATH: metrics_update/data/code_snippets.csv
FIXATIONS_PATH: metrics_update/data/fixations_aoi_relabeled.csv
ANSWERS_PATH: supplementary/answers_experiments.tsv
OUTPUT_PATH: metrics_update/data/base_fixations_design_survey.csv


## Snippets

In [2]:
snip = pd.read_csv(SNIPPETS_PATH)

design = pd.DataFrame(
    {
        "snippet_id": snip["code_snippet"].astype(int),
        "mlcq_sample_id": snip["sample_id"],
        "mlcq_reviewer_id": snip["reviewer_id"],
        "smell_mlcq": snip["smell"].str.strip().str.lower(),
        "severity_mlcq": snip["severity"].str.strip().str.lower(),
        "granularity": snip["type"].str.strip().str.lower(),
        "total_lines": snip["total_lines"],
        "comment_only_lines": snip["comment_only_lines"],
        "blank_lines": snip["blank_lines"],
        "source_code_lines": snip["source_code_lines"],
        "start_line": snip["start_line"].astype(int),
        "end_line": snip["end_line"].astype(int),
    }
)

design = design.sort_values("snippet_id").reset_index(drop=True)

print_head("Design data")
print(f"Snippets: {design["snippet_id"].nunique()}")
display(design)


Design data
------------------------------------------------------------------------------------------
Snippets: 12


,snippet_id,mlcq_sample_id,mlcq_reviewer_id,smell_mlcq,severity_mlcq,granularity,total_lines,comment_only_lines,blank_lines,source_code_lines,start_line,end_line
0,2,8997948,3,long method,minor,function,21,0,0,21,73,93
1,3,7617518,20,data class,minor,class,22,0,6,16,18,39
2,4,7186067,14,feature envy,critical,function,14,0,0,14,103,116
3,5,6299837,10,feature envy,major,function,37,6,1,30,189,225
4,6,6328221,4,feature envy,none,function,18,1,2,15,148,165
5,7,7942242,10,data class,critical,class,36,9,7,20,20,55
6,8,6312810,10,data class,none,class,24,0,5,19,28,51
7,9,8494231,3,long method,major,function,39,1,4,34,573,611
8,10,3922485,3,feature envy,minor,function,14,0,3,11,150,163
9,11,4092105,20,long method,critical,function,27,0,0,27,440,466


## Fixations

In [3]:
fixations = pd.read_csv(FIXATIONS_PATH, sep="\t", dtype={"chunk_num": "Int64"})

# Labels as originally exported (before relabel_aoi.py) are kept as *_raw.
fixations = fixations.rename(
    columns={
        "chunk_original": "chunk_raw",
        "chunk_num_original": "chunk_num_raw",
        "event_original": "event_raw",
    }
)

fixations["participant_id"] = fixations["experiment"]
fixations = fixations.rename(columns={"fixation_target": "snippet_id", "event": "event_original"})

# New event column
fixations["event"] = np.select(
    [
        fixations["event_original"].str.startswith("bottom-up", na=False),
        fixations["event_original"].str.startswith("top-down", na=False)
    ],
    [
        "bottom-up",
        "top-down"
    ],
    default=None,
)

fixations = fixations.merge(
    design[
        ["snippet_id",
        "start_line",
        "end_line"]
    ],
    on="snippet_id",
    how="left",
    validate="many_to_one"
)

# Inside = fixation on one of the chunks of the target code (chunk1..chunkk); "before" and
# "after" are outside. This follows the displayed file (between the START/END markers), unlike
# the MLCQ start_line/end_line, which are offset by the marker lines.
fixations["inside"] = fixations["chunk"].str.fullmatch(r"chunk\d+")

assert fixations["inside"].eq(fixations["event_original"].str.endswith("-inside")).all()

print_head("Fixations data")
print(f"Fixations: {len(fixations)}")
print(f"Participants: {fixations['participant_id'].nunique()}")
print(f"Snippets: {fixations['snippet_id'].nunique()}")

display(
    fixations[
        ["participant_id",
        "snippet_id",
        "fixation_id",
        "source_file_line",
        "start_line",
        "end_line",
        "inside",
        "event_original",
        "event"]
    ].head()
)

Fixations data
------------------------------------------------------------------------------------------
Fixations: 74412
Participants: 25
Snippets: 12


,participant_id,snippet_id,fixation_id,source_file_line,start_line,end_line,inside,event_original,event
0,1,2,43fa7efe-726a-4136-a808-98d9d3c3b5b3,77,73,93,True,top-down-inside,top-down
1,1,2,b3e39d93-8987-41b4-8d3c-cd1323f80451,86,73,93,True,top-down-inside,top-down
2,1,2,b2c5b73a-eea8-43bb-a778-ce9686aa6340,83,73,93,True,bottom-up-inside,bottom-up
3,1,2,e43826ea-2702-47c0-b885-2989535f097b,77,73,93,True,top-down-inside,top-down
4,1,2,cdb9df8b-cd6b-43e4-8d45-83b90d168845,81,73,93,True,top-down-inside,top-down


# Survey

In [4]:
answers = pd.read_csv(ANSWERS_PATH, sep="\t",dtype=str)

answers["participant_id"] = answers["experiment_id"].astype(int)
rows=[]
for snippet_id in range(2,14):
    block = pd.DataFrame(
        {
            "participant_id": answers["participant_id"],
            "snippet_id": snippet_id,
            "perceived_smell_yn": answers.get(f"CodeSnippet{snippet_id}Smell",pd.Series(pd.NA, index=answers.index)),
            "perceived_smell_types_raw": answers.get(f"CodeSnippet{snippet_id}SmellType",pd.Series(pd.NA, index=answers.index)),
            "perceived_severity": answers.get(f"CodeSnippet{snippet_id}SmellSeverity",pd.Series(pd.NA, index=answers.index))
        }
    )
    rows.append(block)

survey = pd.concat(rows, ignore_index=True)

def contains_pattern(value, patterns):
    if not isinstance(value, str):
        return 0
    value = value.lower()
    return int(any(pattern in value for pattern in patterns))

survey["perceived_lm"] = survey["perceived_smell_types_raw"].apply(
    lambda value: contains_pattern(
        value,
        ["long method"],
    )
)

survey["perceived_dc"] = survey["perceived_smell_types_raw"].apply(
    lambda value: contains_pattern(
        value,
        ["data class"],
    )
)

survey["perceived_fe"] = survey["perceived_smell_types_raw"].apply(
    lambda value: contains_pattern(
        value,
        ["feature envy"],
    )
)

perceived_any_named = survey[
    ["perceived_lm", "perceived_dc", "perceived_fe"]
].max(axis=1)

survey["perceived_other"] = (
    survey["perceived_smell_types_raw"].notna()
    & (perceived_any_named == 0)
).astype(int)

print_head("Survey participants × snippet")
print(f"Lines: {len(survey):,}")
print(f"Participants: {survey['participant_id'].nunique()}")
print(f"Snippets: {survey['snippet_id'].nunique()}")

display(survey.head(20))

Survey participants × snippet
------------------------------------------------------------------------------------------
Lines: 312
Participants: 26
Snippets: 12


,participant_id,snippet_id,perceived_smell_yn,perceived_smell_types_raw,perceived_severity,perceived_lm,perceived_dc,perceived_fe,perceived_other
0,1,2,Yes,"Long Method, Feature Envy",Major,1,0,1,0
1,2,2,Yes,Feature Envy,Minor,0,0,1,0
2,3,2,Yes,Long Method,Major,1,0,0,0
3,4,2,No,NaN,NaN,0,0,0,0
4,5,2,Yes,Long Method,Minor,1,0,0,0
5,6,2,Yes,chamadas de métodos subordinados,Minor,0,0,0,1
6,8,2,Yes,Long Method,Critical,1,0,0,0
7,9,2,Yes,Long Method,Minor,1,0,0,0
8,10,2,No,NaN,NaN,0,0,0,0
9,11,2,Yes,"Long Method, Data Class, Feature Envy",Critical,1,1,1,0


# Severity

In [5]:
survey_eval = survey.merge(
    design[
        [
            "snippet_id",
            "smell_mlcq",
            "severity_mlcq",
        ]
    ],
    on="snippet_id",
    how="left",
    validate="many_to_one",
)

smell_column = {
    "long method": "perceived_lm",
    "data class": "perceived_dc",
    "feature envy": "perceived_fe",
}

def correct_identification(row):
    if pd.isna(row["perceived_smell_yn"]):
        return np.nan

    # Clean 
    if row["severity_mlcq"] == "none":
        return int(
            row["perceived_smell_yn"].strip().lower() == "no"
        )

    # Smell
    perceived_column = smell_column.get(row["smell_mlcq"])

    if perceived_column is None:
        return np.nan

    return int(row[perceived_column] == 1)

survey_eval["correct_identification"] = survey_eval.apply(
    correct_identification,
    axis=1,
)

# Severity match
survey_eval["severity_match"] = np.where(
    survey_eval["perceived_severity"]
    .str.strip()
    .str.lower()
    .fillna("")
    == survey_eval["severity_mlcq"],
    1,
    0,
)

survey_columns = [
    "participant_id",
    "snippet_id",
    "perceived_smell_yn",
    "perceived_lm",
    "perceived_dc",
    "perceived_fe",
    "perceived_other",
    "perceived_smell_types_raw",
    "perceived_severity",
    "correct_identification",
    "severity_match",
]

survey_eval = survey_eval[survey_columns]

print_head("Survey evaluation")
display(survey_eval.head(20))


Survey evaluation
------------------------------------------------------------------------------------------


,participant_id,snippet_id,perceived_smell_yn,perceived_lm,perceived_dc,perceived_fe,perceived_other,perceived_smell_types_raw,perceived_severity,correct_identification,severity_match
0,1,2,Yes,1,0,1,0,"Long Method, Feature Envy",Major,1.0,0
1,2,2,Yes,0,0,1,0,Feature Envy,Minor,0.0,1
2,3,2,Yes,1,0,0,0,Long Method,Major,1.0,0
3,4,2,No,0,0,0,0,NaN,NaN,0.0,0
4,5,2,Yes,1,0,0,0,Long Method,Minor,1.0,1
5,6,2,Yes,0,0,0,1,chamadas de métodos subordinados,Minor,0.0,1
6,8,2,Yes,1,0,0,0,Long Method,Critical,1.0,0
7,9,2,Yes,1,0,0,0,Long Method,Minor,1.0,1
8,10,2,No,0,0,0,0,NaN,NaN,0.0,0
9,11,2,Yes,1,1,1,0,"Long Method, Data Class, Feature Envy",Critical,1.0,0


# Final base

In [6]:
# start_line and end_line are already in fixations, so we can drop them 

design_for_merge = design.drop(
    columns=["start_line", "end_line"]
)

base = fixations.merge(
    design_for_merge,
    on="snippet_id",
    how="left",
    validate="many_to_one",
)

base = base.merge(
    survey_eval,
    on=["participant_id", "snippet_id"],
    how="left",
    validate="many_to_one",
)

base = (
    base
    .sort_values(
        [
            "participant_id",
            "snippet_id",
            "fixation_start_event_time",
        ]
    )
    .reset_index(drop=True)
)

print_head("Final base")
print(f"Lines: {len(base):,}")
print(f"Participants: {base['participant_id'].nunique()}")
print(f"Snippets: {base['snippet_id'].nunique()}")

display(base.head(20))

Final base
------------------------------------------------------------------------------------------
Lines: 74,412
Participants: 25
Snippets: 12


,fixation_id,fixation_run_id,fixation_start_event_time,snippet_id,source_file_line,system_time,tracker_time,duration,date_hour,chunk,chunk_num,event_original,event_numeric,experiment,chunk_raw,chunk_num_raw,event_raw,aoi_relabeled,participant_id,event,start_line,end_line,inside,mlcq_sample_id,mlcq_reviewer_id,smell_mlcq,severity_mlcq,granularity,total_lines,comment_only_lines,blank_lines,source_code_lines,perceived_smell_yn,perceived_lm,perceived_dc,perceived_fe,perceived_other,perceived_smell_types_raw,perceived_severity,correct_identification,severity_match
0,43fa7efe-726a-4136-a808-98d9d3c3b5b3,1702677842812,133422869035847173,2,77,1697813303583,1697814541201906,107,2023-10-20T11:48:23.583000000,chunk2,2,top-down-inside,1,1,chunk2,2.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
1,b3e39d93-8987-41b4-8d3c-cd1323f80451,1702677842812,133422869037606325,2,86,1697813303759,1697814541378606,157,2023-10-20T11:48:23.759000000,chunk5,5,top-down-inside,1,1,chunk5,5.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
2,b2c5b73a-eea8-43bb-a778-ce9686aa6340,1702677842812,133422869039674067,2,83,1697813303966,1697814541585207,374,2023-10-20T11:48:23.966000000,chunk5,5,bottom-up-inside,2,1,chunk5,5.0,bottom-up-inside,False,1,bottom-up,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
3,e43826ea-2702-47c0-b885-2989535f097b,1702677842812,133422869044005021,2,77,1697813304400,1697814542018458,97,2023-10-20T11:48:24.400000000,chunk2,2,top-down-inside,1,1,chunk2,2.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
4,cdb9df8b-cd6b-43e4-8d45-83b90d168845,1702677842812,133422869045471106,2,81,1697813304546,1697814542165150,370,2023-10-20T11:48:24.546000000,chunk4,4,top-down-inside,1,1,chunk4,4.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
5,ce1f9c28-c040-4bd6-8601-55ace6dfe6e7,1702677842812,133422869049606083,2,83,1697813304959,1697814542578375,160,2023-10-20T11:48:24.959000000,chunk5,5,top-down-inside,1,1,chunk5,5.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
6,bcda0281-2d47-4c9b-bb51-37161959c9ca,1702677842812,133422869051636985,2,82,1697813305162,1697814542781694,170,2023-10-20T11:48:25.162000000,chunk4,4,top-down-inside,1,1,chunk4,4.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
7,ee67759c-0e1b-4785-9478-f96b697848ef,1702677842812,133422869054002865,2,84,1697813305399,1697814543018315,227,2023-10-20T11:48:25.399000000,chunk5,5,top-down-inside,1,1,chunk5,5.0,top-down-inside,False,1,top-down,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
8,ecbe0ab0-66a5-451f-ad45-103f5f59f176,1702677842812,133422869096446553,2,84,1697813309644,1697814547261093,204,2023-10-20T11:48:29.644000000,chunk5,5,bottom-up-inside,2,1,chunk5,5.0,bottom-up-inside,False,1,bottom-up,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0
9,49ae4a18-6bc2-4d1b-ab48-8f18078edc2c,1702677842812,133422869098898126,2,87,1697813309888,1697814547507785,188,2023-10-20T11:48:29.888000000,chunk5,5,bottom-up-inside,2,1,chunk5,5.0,bottom-up-inside,False,1,bottom-up,73,93,True,8997948,3,long method,minor,function,21,0,0,21,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0


## Organize Columns

In [7]:
first_columns = [
    "participant_id",
    "snippet_id",

    "fixation_id",
    "fixation_run_id",
    "fixation_start_event_time",
    "source_file_line",
    "duration",
    "inside",
    "event",
    "event_original",

    "mlcq_sample_id",
    "mlcq_reviewer_id",
    "smell_mlcq",
    "severity_mlcq",
    "granularity",
    "total_lines",
    "comment_only_lines",
    "blank_lines",
    "source_code_lines",
    "start_line",
    "end_line",

    "perceived_smell_yn",
    "perceived_lm",
    "perceived_dc",
    "perceived_fe",
    "perceived_other",
    "perceived_smell_types_raw",
    "perceived_severity",
    "correct_identification",
    "severity_match",
]

remaining_columns = [
    column
    for column in base.columns
    if column not in first_columns
]

base = base[first_columns + remaining_columns]

display(base.head(20))


,participant_id,snippet_id,fixation_id,fixation_run_id,fixation_start_event_time,source_file_line,duration,inside,event,event_original,mlcq_sample_id,mlcq_reviewer_id,smell_mlcq,severity_mlcq,granularity,total_lines,comment_only_lines,blank_lines,source_code_lines,start_line,end_line,perceived_smell_yn,perceived_lm,perceived_dc,perceived_fe,perceived_other,perceived_smell_types_raw,perceived_severity,correct_identification,severity_match,system_time,tracker_time,date_hour,chunk,chunk_num,event_numeric,experiment,chunk_raw,chunk_num_raw,event_raw,aoi_relabeled
0,1,2,43fa7efe-726a-4136-a808-98d9d3c3b5b3,1702677842812,133422869035847173,77,107,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813303583,1697814541201906,2023-10-20T11:48:23.583000000,chunk2,2,1,1,chunk2,2.0,top-down-inside,False
1,1,2,b3e39d93-8987-41b4-8d3c-cd1323f80451,1702677842812,133422869037606325,86,157,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813303759,1697814541378606,2023-10-20T11:48:23.759000000,chunk5,5,1,1,chunk5,5.0,top-down-inside,False
2,1,2,b2c5b73a-eea8-43bb-a778-ce9686aa6340,1702677842812,133422869039674067,83,374,True,bottom-up,bottom-up-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813303966,1697814541585207,2023-10-20T11:48:23.966000000,chunk5,5,2,1,chunk5,5.0,bottom-up-inside,False
3,1,2,e43826ea-2702-47c0-b885-2989535f097b,1702677842812,133422869044005021,77,97,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813304400,1697814542018458,2023-10-20T11:48:24.400000000,chunk2,2,1,1,chunk2,2.0,top-down-inside,False
4,1,2,cdb9df8b-cd6b-43e4-8d45-83b90d168845,1702677842812,133422869045471106,81,370,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813304546,1697814542165150,2023-10-20T11:48:24.546000000,chunk4,4,1,1,chunk4,4.0,top-down-inside,False
5,1,2,ce1f9c28-c040-4bd6-8601-55ace6dfe6e7,1702677842812,133422869049606083,83,160,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813304959,1697814542578375,2023-10-20T11:48:24.959000000,chunk5,5,1,1,chunk5,5.0,top-down-inside,False
6,1,2,bcda0281-2d47-4c9b-bb51-37161959c9ca,1702677842812,133422869051636985,82,170,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813305162,1697814542781694,2023-10-20T11:48:25.162000000,chunk4,4,1,1,chunk4,4.0,top-down-inside,False
7,1,2,ee67759c-0e1b-4785-9478-f96b697848ef,1702677842812,133422869054002865,84,227,True,top-down,top-down-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813305399,1697814543018315,2023-10-20T11:48:25.399000000,chunk5,5,1,1,chunk5,5.0,top-down-inside,False
8,1,2,ecbe0ab0-66a5-451f-ad45-103f5f59f176,1702677842812,133422869096446553,84,204,True,bottom-up,bottom-up-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813309644,1697814547261093,2023-10-20T11:48:29.644000000,chunk5,5,2,1,chunk5,5.0,bottom-up-inside,False
9,1,2,49ae4a18-6bc2-4d1b-ab48-8f18078edc2c,1702677842812,133422869098898126,87,188,True,bottom-up,bottom-up-inside,8997948,3,long method,minor,function,21,0,0,21,73,93,Yes,1.0,0.0,1.0,0.0,"Long Method, Feature Envy",Major,1.0,0.0,1697813309888,1697814547507785,2023-10-20T11:48:29.888000000,chunk5,5,2,1,chunk5,5.0,bottom-up-inside,False


## Export table

In [8]:
base.to_csv(OUTPUT_PATH, index=False)

print_head("Saved base to CSV")
print(f"Saved base to CSV: {OUTPUT_PATH.relative_to(ROOT_DIR)}")

Saved base to CSV
------------------------------------------------------------------------------------------
Saved base to CSV: metrics_update/data/base_fixations_design_survey.csv
